<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.4-alloydb-bigquery/notebooks/GCP_Capstone_2.4_AlloyDB_BigQuery.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.4 AlloyDB pgvector & BigQuery Vector Search
**Netsetos GenAI Engineering — GCP Capstone**

SQL-native vector search, ScaNN vs HNSW, VECTOR_SEARCH(), and the 3-database decision framework.


## Setup
Note: AlloyDB requires a provisioned instance ($200+/mo). This notebook demonstrates SQL patterns.
BigQuery examples use the free tier (1 TiB/month).


In [ ]:
!pip install -q google-genai google-cloud-bigquery pgvector psycopg2-binary
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS


## Cell 1: AlloyDB pgvector SQL Patterns
These SQL statements run on AlloyDB (not Colab). Study the patterns.


In [ ]:
# AlloyDB SQL patterns (reference — run on AlloyDB, not Colab)
ALLOYDB_SQL = """
-- 1. Enable extensions
CREATE EXTENSION IF NOT EXISTS vector;
CREATE EXTENSION IF NOT EXISTS google_ml_integration CASCADE;
CREATE EXTENSION IF NOT EXISTS alloydb_scann CASCADE;

-- 2. Create table with embedding column
CREATE TABLE documents (
    id SERIAL PRIMARY KEY,
    content TEXT,
    category VARCHAR(50),
    embedding vector(768)
);

-- 3. Create ScaNN index
CREATE INDEX idx_scann ON documents
USING scann (embedding cosine) WITH (num_leaves = 100);
ANALYZE documents;

-- 4. Vector search with cosine distance
SELECT id, content,
       embedding <=> '[0.1, 0.2, ...]'::vector AS distance
FROM documents
ORDER BY embedding <=> '[0.1, 0.2, ...]'::vector
LIMIT 5;

-- 5. Filtered search with JOIN
SELECT d.content, a.name, d.embedding <=> '[...]'::vector AS dist
FROM documents d JOIN authors a ON d.author_id = a.id
WHERE d.category = 'ai_ml'
ORDER BY d.embedding <=> '[...]'::vector LIMIT 5;

-- 6. Auto-embedding with generated column
CREATE TABLE kb (
    id SERIAL PRIMARY KEY,
    content TEXT,
    embedding vector(768) GENERATED ALWAYS AS (
        embedding('text-embedding-005', content)
    ) STORED
);
"""
print(ALLOYDB_SQL)


## Cell 2: AlloyDB Python Connection Pattern


In [ ]:
# Pattern for connecting to AlloyDB from Python
# Requires: pip install psycopg2-binary pgvector

CONNECT_PATTERN = """
import psycopg2
from pgvector.psycopg2 import register_vector
from google import genai
from google.genai import types

# Initialize
ai = genai.Client(vertexai=True, project='PROJECT', location='us-central1')
conn = psycopg2.connect(host='ALLOYDB_IP', dbname='documind', user='postgres', password='***')
register_vector(conn)
cur = conn.cursor()

# Embed + Insert
text = 'AlloyDB provides sub-millisecond vector search.'
emb = ai.models.embed_content(
    model='gemini-embedding-001', contents=text,
    config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768)
).embeddings[0].values

cur.execute('INSERT INTO documents (content, embedding) VALUES (%s, %s)', (text, emb))
conn.commit()

# Search
query_emb = ai.models.embed_content(
    model='gemini-embedding-001', contents='How fast is AlloyDB?',
    config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY', output_dimensionality=768)
).embeddings[0].values

cur.execute('''
    SELECT content, embedding <=> %s::vector AS distance
    FROM documents ORDER BY embedding <=> %s::vector LIMIT 5
''', (str(query_emb), str(query_emb)))

for content, dist in cur.fetchall():
    print(f'  [{1-dist:.4f}] {content}')
"""
print(CONNECT_PATTERN)


## Cell 3: BigQuery VECTOR_SEARCH (Runnable)


In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)

# Create dataset if not exists
bq.create_dataset(f'{PROJECT_ID}.documind_vectors', exists_ok=True)

# BigQuery SQL patterns
BQ_SQL = """
-- 1. Create remote embedding model
CREATE OR REPLACE MODEL `documind_vectors.embed_model`
  REMOTE WITH CONNECTION DEFAULT
  OPTIONS (ENDPOINT = 'text-embedding-005');

-- 2. Generate embeddings
CREATE OR REPLACE TABLE `documind_vectors.doc_embeddings` AS
SELECT * FROM AI.GENERATE_EMBEDDING(
  MODEL `documind_vectors.embed_model`,
  (SELECT 'doc_1' AS id, 'Transformers use self-attention' AS content
   UNION ALL SELECT 'doc_2', 'RAG combines retrieval with generation'
   UNION ALL SELECT 'doc_3', 'AlloyDB is a managed PostgreSQL database')
);

-- 3. Create vector index (requires Enterprise edition)
CREATE OR REPLACE VECTOR INDEX doc_idx
ON `documind_vectors.doc_embeddings`(ml_generate_embedding_result)
OPTIONS (index_type = 'IVF', distance_type = 'COSINE');

-- 4. Search
SELECT base.content, distance
FROM VECTOR_SEARCH(
  TABLE `documind_vectors.doc_embeddings`,
  'ml_generate_embedding_result',
  (SELECT ml_generate_embedding_result AS embedding
   FROM AI.GENERATE_EMBEDDING(
     MODEL `documind_vectors.embed_model`,
     (SELECT 'How does attention work?' AS content))),
  top_k => 3, distance_type => 'COSINE'
);
"""
print(BQ_SQL)


## Cell 4: Decision Framework Calculator


In [ ]:
def recommend_vector_db(num_docs, need_realtime=True, need_sql=False, budget_monthly=0):
    print(f'\nDocuMind Vector DB Recommendation:')
    print(f'  Documents: {num_docs:,}')
    print(f'  Real-time: {need_realtime}')
    print(f'  SQL needed: {need_sql}')
    print(f'  Budget: Rs {budget_monthly:,}/mo\n')
    
    if num_docs < 10000 and not need_sql:
        print('  Firestore (free tier)')
        print('  - find_nearest() with cosine distance')
        print('  - Flat index (exact KNN)')
        print('  - Cost: Rs 0/mo')
    elif need_realtime and (need_sql or num_docs >= 10000):
        print('  AlloyDB with ScaNN')
        print('  - pgvector <=> operator')
        print('  - ScaNN index (1-30ms latency)')
        print('  - Cost: Rs 17,000+/mo')
    else:
        print('  BigQuery VECTOR_SEARCH')
        print('  - AI.GENERATE_EMBEDDING() + VECTOR_SEARCH()')
        print('  - IVF/TreeAH index')
        print('  - Cost: Rs 0-8,500/mo (on-demand)')

# Test scenarios
recommend_vector_db(500, need_realtime=True, need_sql=False)
recommend_vector_db(100000, need_realtime=True, need_sql=True)
recommend_vector_db(5000000, need_realtime=False, need_sql=False)


## Cell 5: Compare Index Types


In [ ]:
index_comparison = {
    'IVFFlat': {'algorithm': 'K-means', 'build': 'Fast', 'memory': 'Medium',
                'in_memory_latency': 'Good', 'out_of_memory': 'OK', 'availability': 'Any PostgreSQL'},
    'HNSW': {'algorithm': 'Graph', 'build': 'Slow', 'memory': 'Largest',
             'in_memory_latency': 'Great', 'out_of_memory': 'Degrades badly', 'availability': 'Any PostgreSQL'},
    'ScaNN': {'algorithm': 'Tree+quantization', 'build': '10x faster', 'memory': '4x smaller',
              'in_memory_latency': 'Up to 4x better', 'out_of_memory': '10x better', 'availability': 'AlloyDB only'},
    'IVF (BQ)': {'algorithm': 'K-means', 'build': 'Auto', 'memory': 'Serverless',
                  'in_memory_latency': 'Seconds', 'out_of_memory': 'N/A', 'availability': 'BigQuery'},
    'TreeAH (BQ)': {'algorithm': 'Tree+hashing', 'build': '10x faster', 'memory': 'Serverless',
                     'in_memory_latency': 'Batch optimized', 'out_of_memory': 'N/A', 'availability': 'BigQuery'},
}

print(f"{'Index':<15} {'Algorithm':<20} {'Build':<12} {'Memory':<12} {'Availability':<18}")
print('-' * 80)
for name, props in index_comparison.items():
    print(f"{name:<15} {props['algorithm']:<20} {props['build']:<12} {props['memory']:<12} {props['availability']:<18}")


## ✅ Module 2 Complete!

- ✅ 2.1: Token Economics — tokenization, multilingual costs, context budgeting
- ✅ 2.2: Embeddings — gemini-embedding-001, cosine similarity, task types
- ✅ 2.3: Firestore Vector Search — find_nearest(), complete RAG pipeline
- ✅ 2.4: AlloyDB & BigQuery — SQL vectors, ScaNN/HNSW, decision framework

**Next: Module 3 — Prompt Engineering & Structured Output**
